## 第 3 周练习 —— 合成数据集生成器（Synthetic Dataset Generator）

本笔记本演示：只用 **Anthropic API Key**，通过 Messages API 让 Claude 生成「看起来真实、实为虚构」的 JSON 数据集，并用 **Gradio** 提供简易界面。


### 用 LLM（Anthropic API）生成合成数据集

**练习目标**：全部流程只需 Anthropic API Key——无需下载本地模型。

对应第 3 周常见主题：调用云端 LLM、构造结构化输出（JSON）、用 Gradio 做最小可用界面。


In [ ]:
# ========== 导入与 API 准备：后续请求与 UI 依赖 ==========

# 导入 os：从环境变量读取 API_KEY
import os
# 导入 json：序列化请求体、解析/保存生成的 JSON
import json
# 导入 requests：用 HTTP 调用 Anthropic REST API
import requests
# 导入 gradio：搭「主题 / 条数 / 模型」生成界面
import gradio as gr
# 从 dotenv 导入 load_dotenv：把 .env 读进环境变量
from dotenv import load_dotenv


In [ ]:
# ========== 从 .env 加载 Anthropic API Key ==========

# 从 .env 加载变量（默认不 override；保持原调用方式）
load_dotenv()

# 读取 Anthropic API Key：环境变量名必须是 API_KEY
API_KEY = os.getenv("API_KEY")

# 若缺失密钥则立刻失败，避免后面请求带着空 key 难排查
if not API_KEY:
    # 错误文案影响行为判断，保持英文原样
    raise ValueError(" API_KEY not found. Check your .env file")

# 成功提示（打印文案保持原样）
print("API key loaded successfully!")


API key loaded successfully!


In [ ]:
# ========== Anthropic 接口：Messages URL + 列出可用模型 ==========

# Anthropic 接口地址：后面 POST 生成数据时用这个 Messages 端点
API_URL = "https://api.anthropic.com/v1/messages"

# 查看当前账号可访问的模型列表
# GET /v1/models；headers 需 x-api-key 与 anthropic-version
r = requests.get(
    "https://api.anthropic.com/v1/models",
    headers={
        "x-api-key": API_KEY,
        "anthropic-version": "2023-06-01"
    },
)
# 成功则打印 JSON；否则打印原始错误文本，便于排查权限/密钥问题
print(r.json() if r.ok else r.text)


{'data': [{'type': 'model', 'id': 'claude-haiku-4-5-20251001', 'display_name': 'Claude Haiku 4.5', 'created_at': '2025-10-15T00:00:00Z'}, {'type': 'model', 'id': 'claude-sonnet-4-5-20250929', 'display_name': 'Claude Sonnet 4.5', 'created_at': '2025-09-29T00:00:00Z'}, {'type': 'model', 'id': 'claude-opus-4-1-20250805', 'display_name': 'Claude Opus 4.1', 'created_at': '2025-08-05T00:00:00Z'}, {'type': 'model', 'id': 'claude-opus-4-20250514', 'display_name': 'Claude Opus 4', 'created_at': '2025-05-22T00:00:00Z'}, {'type': 'model', 'id': 'claude-sonnet-4-20250514', 'display_name': 'Claude Sonnet 4', 'created_at': '2025-05-22T00:00:00Z'}, {'type': 'model', 'id': 'claude-3-7-sonnet-20250219', 'display_name': 'Claude Sonnet 3.7', 'created_at': '2025-02-24T00:00:00Z'}, {'type': 'model', 'id': 'claude-3-5-haiku-20241022', 'display_name': 'Claude Haiku 3.5', 'created_at': '2024-10-22T00:00:00Z'}, {'type': 'model', 'id': 'claude-3-haiku-20240307', 'display_name': 'Claude Haiku 3', 'created_at': '

In [ ]:
# ========== 可选模型表：UI 显示名 -> Anthropic model id ==========
# 用于对比的模型（覆盖不同档位）

MODELS = {
    # 快且便宜：适合演示与小规模合成数据
    "Claude 3 Haiku": "claude-3-haiku-20240307",     # 快且便宜
    # 较新的 Haiku 档
    "Claude Haiku 4.5": "claude-haiku-4-5-20251001",
    # 较新的 Sonnet 档
    "Claude Sonnet 4.5": "claude-sonnet-4-5-20250929",     # 快且便宜
    # Opus 4.1：更强、通常更贵
    "Claude Opus 4.1": "claude-opus-4-1-20250805",
    # Opus 4
    "Claude Opus 4": "claude-opus-4-20250514",     # 快且便宜
    # Sonnet 4：均衡
    "Claude Sonnet 4": "claude-sonnet-4-20250514",   # 均衡
    # Sonnet 3.7：更强（通常更慢）
    "Claude Sonnet 3.7": "claude-3-7-sonnet-20250219"        # 更强（通常更慢）
}


## 合成数据集生成函数

下面定义 `generate_dataset`：拼英文 prompt → POST Messages API → 取出模型返回的文本（期望是 JSON 数组）。


In [ ]:
# ========== 数据集生成器：调用 Claude 产出 JSON 数组文本 ==========

def generate_dataset(topic, n_records, model_choice):
    # 构造 user prompt：要求只输出合法 JSON 数组（prompt 原文勿改）
    prompt = f"""
You are a data generator creating synthetic datasets.
Generate {n_records} records about {topic}.
Output only a valid JSON array (no explanations or markdown).
Each record should have 4–6 fields and look realistic but fake.
"""

    # Anthropic REST 所需请求头：密钥、JSON、API 版本
    headers = {
        "x-api-key": API_KEY,
        "content-type": "application/json",
        "anthropic-version": "2023-06-01",
    }

    # 请求体：模型 id、生成上限、温度、单轮 user 消息
    payload = {
        "model": model_choice,
        "max_tokens": 500,
        "temperature": 0.7,
        "messages": [{"role": "user", "content": prompt}],
    }

    # POST 到 Messages API；body 用 json.dumps 序列化
    response = requests.post(API_URL, headers=headers, data=json.dumps(payload))
    # 解析响应 JSON
    result = response.json()

    # Anthropic 成功响应通常含 content 列表；取第一块 text
    if "content" in result and len(result["content"]) > 0:
        return result["content"][0]["text"]
    else:
        # 失败时把整个 result 拼进 Error 字符串返回（文案格式保持原样）
        return f"Error: {result}"


## Gradio 界面

用 Blocks 做一个最小演示：输入主题与条数、选择模型、点击生成，在代码框里看 JSON 输出。


In [ ]:
# ========== 简单的 Gradio 界面，用于生成数据集 ==========

def ui_generate(topic, n_records, model_label):
    # 把下拉显示名映射成真正的 model id
    model_id = MODELS[model_label]
    # 演示用途：限制条数，最多 5，控制费用与响应长度
    n_records = min(int(n_records), 5)  # 演示用途：限制条数
    # 调用上面的生成函数，返回 JSON 文本给输出组件
    return generate_dataset(topic, n_records, model_id)

# Gradio Blocks：用 css 限制容器最大宽度并居中
with gr.Blocks(css=".gradio-container {max-width: 600px !important; margin: auto;}") as demo:
    # 标题 Markdown（界面文案原文保留）
    gr.Markdown("## Synthetic Dataset Generator using LLM APIs (Claude)")

    with gr.Row():
        # 数据集主题输入；默认 Employee Records
        topic = gr.Textbox(label="Dataset Topic", value="Employee Records")
        # 生成条数；演示上限在 ui_generate 里再裁到 5
        n_records = gr.Number(label="Number of Records (Max 5 for demo purposes)", value=3)

    # 模型下拉：选项为 MODELS 的键，默认 Claude 3 Haiku
    model_choice = gr.Dropdown(
        label="Choose Model",
        choices=list(MODELS.keys()),
        value="Claude 3 Haiku"
    )

    # 生成按钮（标签含 emoji，保持原样）
    btn = gr.Button("🚀 Generate")

    # 可滚动的紧凑输出区：用 Code 组件按 JSON 高亮展示
    output = gr.Code(label="Generated JSON Dataset", language="json", lines=15, interactive=False)

    # 点击按钮：把三个输入传给 ui_generate，结果写入 output
    btn.click(ui_generate, inputs=[topic, n_records, model_choice], outputs=[output])

# 启动本地 Gradio 服务
demo.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## 将输出保存到文件

可选：把界面里得到的字符串解析为 JSON 后写入磁盘；若不是合法 JSON，则按纯文本保存。


In [ ]:
# ========== 保存生成结果：优先 JSON，失败则纯文本 ==========

def save_dataset_to_file(data, filename="synthetic_dataset.json"):
    try:
        # 尝试把模型输出解析成 Python 对象（期望 list/dict）
        parsed = json.loads(data)
    except:
        # 不是合法 JSON：按纯文本写入（提示文案保持原样）
        print("Not valid JSON, saving as plain text instead.")
        with open(filename, "w", encoding="utf-8") as f:
            f.write(data)
        return

    # 合法 JSON：美化缩进后写回文件
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(parsed, f, indent=2)
    # 成功提示（格式字符串保持原样）
    print(f"Dataset saved as {filename}")
